# JAX-ALFA Simulation Run Log

Scans every `run.log` file under `examples*/` and reports when each simulation
was run, how long it took, and on which platform. Sort by **Completed** ascending
to identify the oldest (potentially stale) runs.

**Platform** is the results tree: `examples` (UAlbany DGX, A100), `examples_A6000ada` (HAL, RTX 6000 Ada), `examples_H100` (Empire AI, H100).

In [1]:
from IPython.display import display, Markdown
from datetime import datetime, timezone
display(Markdown(f"*Last run: {datetime.now(timezone.utc).strftime('%B %d, %Y at %H:%M UTC')}*"))

*Last run: November 19, 2026 at 11:21 UTC*

In [2]:
import os, re
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML

def find_repo_root():
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'examples').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate jaxalfa repository root')

def read_log_info(log_path, tail_bytes=16384):
    """Read elapsed time and CFL values from the tail of a run.log."""
    info = {'elapsed': None, 'cfl_last': None, 'cfl_max': None}
    try:
        size = os.path.getsize(log_path)
        with open(log_path, 'rb') as f:
            f.seek(max(0, size - tail_bytes))
            tail = f.read().decode('utf-8', errors='ignore')
        m = re.search(r'Total Elapsed Time:\s+([\d.]+)\s+seconds', tail)
        if m:
            info['elapsed'] = float(m.group(1))
        for m in re.finditer(r'Current CFL:\s+([\d.]+)', tail):
            info['cfl_last'] = float(m.group(1))
        for m in re.finditer(r'CFLmax:\s+([\d.]+)', tail):
            info['cfl_max'] = float(m.group(1))
    except Exception:
        pass
    return info

repo = find_repo_root()
print(f'Repository root: {repo}')

Repository root: /Users/sukantabasu/Dropbox/Codes/LES/JAX-ALFA/jaxalfa


In [3]:
rows = []

for log_path in sorted(repo.glob('examples*/*/runs/*/run.log')):
    rel      = log_path.relative_to(repo).parts
    platform = rel[0]
    case     = rel[1]
    run      = rel[3]

    mtime     = os.path.getmtime(log_path)
    completed = datetime.fromtimestamp(mtime, tz=timezone.utc)
    info      = read_log_info(log_path)
    elapsed   = info['elapsed']

    if elapsed is not None:
        started   = completed - timedelta(seconds=elapsed)
        wall_time = str(timedelta(seconds=int(elapsed)))
    else:
        started   = None
        wall_time = '—'

    rows.append({
        'Platform'       : platform,
        'Case'           : case,
        'Run'            : run,
        'Started (UTC)'  : started.strftime('%Y-%m-%d %H:%M') if started else '—',
        'Completed (UTC)': completed.strftime('%Y-%m-%d %H:%M'),
        'Wall Time'      : wall_time,
        'CFL (last)'     : f"{info['cfl_last']:.4f}" if info['cfl_last'] is not None else '—',
        'CFL (max)'      : f"{info['cfl_max']:.4f}"  if info['cfl_max']  is not None else '—',
        '_completed_dt'  : completed,
    })

df_all = pd.DataFrame(rows)
print(f'{len(df_all)} runs found across {df_all["Platform"].nunique()} platform(s)')

51 runs found across 5 platform(s)


## By Completion Date (oldest first)

Oldest runs appear at the top — these are candidates for re-running with the latest model.

In [4]:
display_cols = ['Platform', 'Case', 'Run', 'Started (UTC)', 'Completed (UTC)', 'Wall Time', 'CFL (last)', 'CFL (max)']

df_by_date = (
    df_all.sort_values('_completed_dt')
          .reset_index(drop=True)
          [display_cols]
)
df_by_date.index += 1
display(df_by_date.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time,CFL (last),CFL (max)
1,examples_A6000,CBL_N91,128x128x128_LASDD_SM_SP,2026-05-22 22:04,2026-05-22 23:04,1:00:11,0.0350,0.0410
2,examples_sandbox_A6000ada,SBL_GABLS1,64x64x64_LASDD_SM_DP,2026-05-31 11:42,2026-05-31 12:32,0:49:19,0.1740,0.2510
3,examples_sandbox_A6000ada,SBL_GABLS1,128x128x128_LASDD_SM_DP,2026-05-31 11:54,2026-06-01 00:22,12:28:34,0.1940,0.2680
4,examples_sandbox,NBL_Profile,NBL_Profile_R4,2026-06-30 13:29,2026-07-03 04:31,"2 days, 15:02:08",0.1290,0.1430
5,examples_sandbox,NBL_Profile,NBL_Profile_R3,2026-06-30 13:29,2026-07-03 04:38,"2 days, 15:09:18",0.1300,0.1480
6,examples_sandbox,NBL_Profile,NBL_Profile_R2,2026-06-30 13:29,2026-07-04 01:16,"3 days, 11:47:37",0.1260,0.1410
7,examples_sandbox,NBL_Profile,NBL_Profile_R1,2026-06-30 13:29,2026-07-04 01:24,"3 days, 11:55:21",0.1290,0.1470
8,examples_sandbox,NBL_Profile,NBL_Profile_R5,2026-06-30 13:29,2026-07-05 05:20,"4 days, 15:50:40",0.1350,0.1560
9,examples_sandbox,NBL_Profile,NBL_Profile_R6,2026-06-30 13:29,2026-07-05 05:24,"4 days, 15:55:02",0.1290,0.1460
10,examples_sandbox,NBL_Profile,NBL_Profile_R8,2026-07-20 21:42,2026-07-25 20:25,"4 days, 22:42:49",0.1340,0.1410


## By Platform → Case → Run

Same data grouped alphabetically by platform, case, and run name.

In [5]:
df_by_name = (
    df_all.sort_values(['Platform', 'Case', 'Run'])
          .reset_index(drop=True)
          [display_cols]
)
df_by_name.index += 1
display(df_by_name.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time,CFL (last),CFL (max)
1,examples,CBL_N91,64x64x64_LAD_SM_DP,2026-11-19 09:36,2026-11-19 09:44,0:07:49,0.0130,0.0140
2,examples,CBL_N91,64x64x64_LAD_SM_SP,2026-11-19 09:26,2026-11-19 09:34,0:07:52,0.0120,0.0140
3,examples,CBL_N91,64x64x64_LASDD_SM_DP,2026-11-19 06:51,2026-11-19 07:01,0:10:05,0.0120,0.0160
4,examples,CBL_N91,64x64x64_LASDD_SM_SP,2026-11-19 06:51,2026-11-19 07:01,0:09:58,0.0130,0.0150
5,examples,CBL_N91,64x64x64_LASDD_WL_DP,2026-11-19 08:01,2026-11-19 08:09,0:07:52,0.0120,0.0140
6,examples,CBL_N91,64x64x64_LASDD_WL_SP,2026-11-19 07:56,2026-11-19 08:04,0:08:06,0.0130,0.0160
7,examples,DC_Wangara,64x64x64_LAD_SM_DP,2026-11-19 10:42,2026-11-19 11:11,0:29:28,0.0520,0.0860
8,examples,DC_Wangara,64x64x64_LAD_SM_SP,2026-11-19 10:37,2026-11-19 11:10,0:33:01,0.0500,0.0860
9,examples,DC_Wangara,64x64x64_LASDD_SM_DP,2026-11-19 07:26,2026-11-19 08:04,0:37:25,0.0500,0.0870
10,examples,DC_Wangara,64x64x64_LASDD_SM_SP,2026-11-19 07:21,2026-11-19 08:00,0:38:29,0.0510,0.0980


## Summary by Platform

In [6]:
summary = (
    df_all.groupby('Platform')
          .agg(
              Runs        = ('Run', 'count'),
              Oldest_Run  = ('_completed_dt', lambda x: x.min().strftime('%Y-%m-%d')),
              Newest_Run  = ('_completed_dt', lambda x: x.max().strftime('%Y-%m-%d')),
          )
          .rename(columns={'Oldest_Run': 'Oldest Run', 'Newest_Run': 'Newest Run'})
)
display(summary)

,Runs,Oldest Run,Newest Run
Platform,,,
examples,32,2026-11-19,2026-11-19
examples_A6000,1,2026-05-22,2026-05-22
examples_A6000ada,8,2026-11-19,2026-11-19
examples_sandbox,8,2026-07-03,2026-07-27
examples_sandbox_A6000ada,2,2026-05-31,2026-06-01
